In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
REPO = Path.home() / "Code" / "processing" / "riboseq"
PROJECT = Path("/mnt/d/Ibnu/riboseq/AT")

NFCORE_RUN = PROJECT / "nfcore" / "10pct"
SALMON_DIR = NFCORE_RUN / "quantification" / "salmon"
TABLES = PROJECT / "TABLES"

TRANSCRIPT_COUNTS = SALMON_DIR / "salmon.merged.transcript_counts.tsv"
TRANSCRIPT_TPM = SALMON_DIR / "salmon.merged.transcript_tpm.tsv"
TRANSCRIPT_LENGTHS = SALMON_DIR / "salmon.merged.transcript_lengths.tsv"

SALMON_OUT = TABLES / "salmon_transcript.10pct.tsv"

In [3]:
counts = pd.read_csv(TRANSCRIPT_COUNTS, sep="\t")
tpm = pd.read_csv(TRANSCRIPT_TPM, sep="\t")
lengths = pd.read_csv(TRANSCRIPT_LENGTHS, sep="\t")

print(counts.shape)
print(tpm.shape)
print(lengths.shape)

counts.head()

(59051, 4)
(59051, 4)
(59051, 4)


,tx,gene_id,RIBO10,RNA10
0,AT1G01010.1,AT1G01010,2.0,19.000
1,AT1G01020.2,AT1G01020,0.0,24.875
2,AT1G01020.6,AT1G01020,0.0,0.000
3,AT1G01020.1,AT1G01020,0.0,68.125
4,AT1G01020.3,AT1G01020,0.0,0.000


In [4]:
# ============================================
# Standardize Salmon tables
# ============================================

counts2 = counts.rename(columns={
    "tx": "Transcript_ID",
    "gene_id": "Gene_ID",
    "RIBO10": "Ribo_NumReads",
    "RNA10": "RNA_NumReads",
})

tpm2 = tpm.rename(columns={
    "tx": "Transcript_ID",
    "gene_id": "Gene_ID",
    "RIBO10": "Ribo_TPM",
    "RNA10": "RNA_TPM",
})

lengths2 = lengths.rename(columns={
    "tx": "Transcript_ID",
    "gene_id": "Gene_ID",
    "RIBO10": "Ribo_Length",
    "RNA10": "RNA_Length",
})

In [9]:
# ============================================
# Merge Salmon quantification
# ============================================

salmon = (
    counts2
    .merge(
        tpm2[["Transcript_ID", "Ribo_TPM", "RNA_TPM"]],
        on="Transcript_ID",
        how="left",
    )
    .merge(
        lengths2[["Transcript_ID", "Ribo_Length", "RNA_Length"]],
        on="Transcript_ID",
        how="left",
    )
)

print(salmon.head())
print(salmon.shape)
# ============================================
# Validate Salmon table
# ============================================

print("Shape:", salmon.shape)
print()
print("Missing values:")
print(salmon.isna().sum())
print()
print("Duplicated Transcript_ID:", salmon["Transcript_ID"].duplicated().sum())

  Transcript_ID    Gene_ID  Ribo_NumReads  RNA_NumReads  Ribo_TPM    RNA_TPM  \
0   AT1G01010.1  AT1G01010            2.0        19.000  0.111252   3.472288   
1   AT1G01020.2  AT1G01020            0.0        24.875  0.000000   7.810271   
2   AT1G01020.6  AT1G01020            0.0         0.000  0.000000   0.000000   
3   AT1G01020.1  AT1G01020            0.0        68.125  0.000000  16.592177   
4   AT1G01020.3  AT1G01020            0.0         0.000  0.000000   0.000000   

   Ribo_Length  RNA_Length  
0       1438.0      1438.0  
1        837.0       837.0  
2        694.0       694.0  
3       1079.0      1079.0  
4       1170.0      1170.0  
(59051, 8)
Shape: (59051, 8)

Missing values:
Transcript_ID    0
Gene_ID          0
Ribo_NumReads    0
RNA_NumReads     0
Ribo_TPM         0
RNA_TPM          0
Ribo_Length      0
RNA_Length       0
dtype: int64

Duplicated Transcript_ID: 0


In [10]:
# ============================================
# Save Salmon table
# ============================================

salmon.to_csv(SALMON_OUT, sep="\t", index=False)

print(SALMON_OUT)
print("Saved:", SALMON_OUT.exists())

/mnt/d/Ibnu/riboseq/AT/TABLES/salmon_transcript.10pct.tsv
Saved: True
